In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
# ===================================================================
# STEP 0: Install the CORRECT Offline RL Library Version
# ===================================================================
# This is the most important step. We install version 1.1.1
# to match the code.
!pip install d3rlpy==1.1.1 -q

import pandas as pd
import numpy as np
import os
import d3rlpy

# --- We use the ORIGINAL imports for version 1.1.1 ---
from d3rlpy.algos.discrete import CQL 
from d3rlpy.dataset import OfflineDataset
# --- End of Fix ---

from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"d3rlpy version: {d3rlpy.__version__}") # Should print 1.1.1

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

# This code block automatically finds your input files
input_dir = '/kaggle/input/preprocess'


# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the RL Dataset (s, a, r, t)
# ===================================================================
print("\nEngineering RL dataset...")

def calculate_rewards(df_rewards, y_outcomes):
    """Calculates profit/loss for each loan."""
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0: # Fully Paid
            reward = loan_amnt * int_rate_float
        else: # Defaulted
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# 1. States (s): The applicant's features (scaled)
rl_observations_train = X_train_scaled.values.astype(np.float32)

# 2. Actions (a): The *historical* action. (Always 1)
rl_actions_train = np.ones(len(rl_observations_train), dtype=np.int32)

# 3. Rewards (r): The *observed outcome* from taking action 1.
rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train)

# 4. Terminals (t): Each loan decision is its own "episode".
rl_terminals_train = np.ones(len(rl_observations_train), dtype=np.float32)

# --- Create the d3rlpy OfflineDataset ---
# This is the correct object for d3rlpy v1.1.1
dataset = OfflineDataset(
    observations=rl_observations_train,
    actions=rl_actions_train,
    rewards=rl_rewards_train,
    terminals=rl_terminals_train
)
print("RL Dataset created successfully.")

# ===================================================================
# STEP 3: Build and Train the Offline RL Agent (CQL)
# ===================================================================

# This is the correct way to initialize CQL in v1.1.1
print("\nConfiguring CQL model...")
cql = CQL(n_actions=2, # {0: Deny, 1: Approve}
          scaler='standard', # Automatically scale observations
          use_gpu=True) # Use the available GPU

print("Starting RL agent training (this may take 5-10 minutes)...")

cql.fit(dataset, n_steps=50000)

print("RL agent training complete.")

# ===================================================================
# STEP 4: Evaluate the RL Agent's Policy (Task 3 Requirement)
# ===================================================================
print("\nEvaluating RL agent's learned policy...")

# 1. Get the agent's decisions (policy) for the test set
rl_policy_actions = cql.predict(X_test_scaled.values.astype(np.float32))

# 2. Get the *actual* real-world rewards from the test set
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# 3. Calculate the total profit/loss of the policy
total_policy_value = 0
for i in range(len(rl_policy_actions)):
    action_taken_by_agent = rl_policy_actions[i]
    actual_outcome_reward = rl_rewards_test[i]
    
    if action_taken_by_agent == 1: # If agent CHOSE to approve
        total_policy_value += actual_outcome_reward
    # If agent_action == 0 (Deny), reward is 0.

# The average profit/loss per loan decision
estimated_policy_value = total_policy_value / len(rl_policy_actions)

# --- Analysis ---
num_approved = np.sum(rl_policy_actions == 1)
num_denied = np.sum(rl_policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] RL Agent Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the RL agent's decisions
cm = confusion_matrix(y_test, rl_policy_actions)
print("\n  RL Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by RL)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 6.6 MB/s eta 0:00:0000:01
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'd3rlpy.algos.discrete'

In [2]:
!pip install d3rlpy -q

In [7]:
# ===================================================================
# STEP 0: Install the LATEST Offline RL Library
# ===================================================================
!pip install d3rlpy -q

import pandas as pd
import numpy as np
import os
import d3rlpy

# --- Imports for the LATEST d3rlpy version ---
from d3rlpy.algos import CQLConfig
from d3rlpy.dataset import Episode, ReplayBuffer  # This is the correct import
# --- End of Fix ---

from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"d3rlpy version: {d3rlpy.__version__}") # Should be v2.x.x

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

input_dir = '/kaggle/input/preprocess/'

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the RL Dataset (s, a, r, t)
# ===================================================================
print("\nEngineering RL dataset...")

def calculate_rewards(df_rewards, y_outcomes):
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0:
            reward = loan_amnt * int_rate_float
        else:
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# 1. States (s)
rl_observations_train = X_train_scaled.values.astype(np.float32)

# 2. Actions (a)
rl_actions_train = np.ones(len(rl_observations_train), dtype=np.int32)

# 3. Rewards (r)
rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train)

# 4. Terminals (t) - d3rlpy v2.x calls this 'terminated'
rl_terminated_train = np.ones(len(rl_observations_train), dtype=np.float32)

# --- Create d3rlpy Episode and ReplayBuffer ---
# This is the new way to create a dataset
print("Building Episode...")
episode = Episode(
    observations=rl_observations_train,
    actions=rl_actions_train,
    rewards=rl_rewards_train,
    terminated=rl_terminated_train 
)

print("Building ReplayBuffer...")
buffer = ReplayBuffer.create_empty(
    max_size=len(episode), # Set buffer size to data size
    device='cuda:0' # Use the first GPU
)
buffer.append_episode(episode)

print("RL Dataset (ReplayBuffer) created successfully.")

# ===================================================================
# STEP 3: Build and Train the Offline RL Agent (CQL)
# ===================================================================

print("\nConfiguring CQL model...")
cql_config = CQLConfig(
    scaler='standard'  # Automatically scale observations
)
# We create the model, specifying the GPU
cql = cql_config.create(device='cuda:0') 

print("Starting RL agent training (this may take 5-10 minutes)...")

# We fit on the ReplayBuffer, not the 'dataset' object
cql.fit(buffer, n_steps=50000)

print("RL agent training complete.")

# ===================================================================
# STEP 4: Evaluate the RL Agent's Policy (Task 3 Requirement)
# ===================================================================
print("\nEvaluating RL agent's learned policy...")

# 1. Get the agent's decisions
rl_policy_actions = cql.predict(X_test_scaled.values.astype(np.float32))

# 2. Get the *actual* real-world rewards
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# 3. Calculate the total profit/loss of the policy
total_policy_value = 0
for i in range(len(rl_policy_actions)):
    action_taken_by_agent = rl_policy_actions[i]
    actual_outcome_reward = rl_rewards_test[i]
    
    if action_taken_by_agent == 1: # If agent CHOSE to approve
        total_policy_value += actual_outcome_reward

estimated_policy_value = total_policy_value / len(rl_policy_actions)

# --- Analysis ---
num_approved = np.sum(rl_policy_actions == 1)
num_denied = np.sum(rl_policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] RL Agent Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the RL agent's decisions
cm = confusion_matrix(y_test, rl_policy_actions)
print("\n  RL Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by RL)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

d3rlpy version: 2.8.1
Loaded 141612 training and 35404 test observations.

Engineering RL dataset...
Building Episode...
Building ReplayBuffer...


AttributeError: type object 'ReplayBuffer' has no attribute 'create_empty'

In [10]:
# ===================================================================
# STEP 0: Install the LATEST Offline RL Library
# ===================================================================
!pip install d3rlpy -q

import pandas as pd
import numpy as np
import os
import d3rlpy

# --- Imports for d3rlpy v2.8.1 ---
from d3rlpy.algos import CQLConfig
from d3rlpy.dataset import MDPDataset, Episode
# --- End of imports ---

from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"d3rlpy version: {d3rlpy.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

input_dir = '/kaggle/input/preprocess/'

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the RL Dataset (s, a, r, t)
# ===================================================================
print("\nEngineering RL dataset...")

def calculate_rewards(df_rewards, y_outcomes):
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0:
            reward = loan_amnt * int_rate_float
        else:
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# 1. States (s)
rl_observations_train = X_train_scaled.values.astype(np.float32)

# 2. Actions (a) - d3rlpy v2.8.1 expects actions as 2D array
rl_actions_train = np.ones((len(rl_observations_train), 1), dtype=np.float32)

# 3. Rewards (r)
rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train).astype(np.float32)

# 4. Terminals (t)
rl_terminated_train = np.ones(len(rl_observations_train), dtype=bool)

# --- Create d3rlpy MDPDataset directly from arrays ---
print("Building MDPDataset directly from arrays...")

# MDPDataset in v2.8.1 takes raw arrays directly
dataset = MDPDataset(
    observations=rl_observations_train,
    actions=rl_actions_train,
    rewards=rl_rewards_train,
    terminals=rl_terminated_train  # Note: 'terminals' not 'terminated'
)

print("RL Dataset (MDPDataset) created successfully.")

# ===================================================================
# STEP 3: Build and Train the Offline RL Agent (CQL)
# ===================================================================

print("\nConfiguring CQL model...")
cql_config = CQLConfig(
    observation_scaler='standard'  # Automatically scale observations
)
# Create the model, specifying the GPU
cql = cql_config.create(device='cuda:0') 

print("Starting RL agent training (this may take 5-10 minutes)...")

# Fit on the MDPDataset
cql.fit(
    dataset,
    n_steps=50000,
    show_progress=True
)

print("RL agent training complete.")

# ===================================================================
# STEP 4: Evaluate the RL Agent's Policy (Task 3 Requirement)
# ===================================================================
print("\nEvaluating RL agent's learned policy...")

# 1. Get the agent's decisions
# Prepare test data in the same format
X_test_prepared = X_test_scaled.values.astype(np.float32)
rl_policy_actions_continuous = cql.predict(X_test_prepared)

# Convert continuous actions to discrete (0 or 1)
# Assuming action > 0.5 means approve (1), else deny (0)
rl_policy_actions = (rl_policy_actions_continuous.flatten() > 0.5).astype(int)

# 2. Get the *actual* real-world rewards
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# 3. Calculate the total profit/loss of the policy
total_policy_value = 0
for i in range(len(rl_policy_actions)):
    action_taken_by_agent = rl_policy_actions[i]
    actual_outcome_reward = rl_rewards_test[i]
    
    if action_taken_by_agent == 1:  # If agent CHOSE to approve
        total_policy_value += actual_outcome_reward

estimated_policy_value = total_policy_value / len(rl_policy_actions)

# --- Analysis ---
num_approved = np.sum(rl_policy_actions == 1)
num_denied = np.sum(rl_policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] RL Agent Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the RL agent's decisions
cm = confusion_matrix(y_test, rl_policy_actions)
print("\n  RL Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by RL)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

d3rlpy version: 2.8.1
Loaded 141612 training and 35404 test observations.

Engineering RL dataset...
Building MDPDataset directly from arrays...
2025-10-30 06:47.28 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(36,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-10-30 06:47.28 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-10-30 06:47.28 [info     ] Action size has been automatically determined. action_size=2
RL Dataset (MDPDataset) created successfully.

Configuring CQL model...
Starting RL agent training (this may take 5-10 minutes)...
2025-10-30 06:47.29 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(36,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)

AssertionError: The action-space of the given dataset is not compatible with the algorithm. Please use discrete action-space algorithms. The algorithms list is available below.
https://d3rlpy.readthedocs.io/en/v2.8.1/references/algos.html

In [11]:
# ===================================================================
# STEP 0: Install the LATEST Offline RL Library
# ===================================================================
!pip install d3rlpy -q

import pandas as pd
import numpy as np
import os
import d3rlpy

# --- Imports for d3rlpy v2.8.1 ---
from d3rlpy.algos import DiscreteCQLConfig  # Use Discrete CQL for discrete actions
from d3rlpy.dataset import MDPDataset, Episode
# --- End of imports ---

from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"d3rlpy version: {d3rlpy.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

input_dir = '/kaggle/input/preprocess/'

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the RL Dataset (s, a, r, t)
# ===================================================================
print("\nEngineering RL dataset...")

def calculate_rewards(df_rewards, y_outcomes):
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0:
            reward = loan_amnt * int_rate_float
        else:
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# 1. States (s)
rl_observations_train = X_train_scaled.values.astype(np.float32)

# 2. Actions (a) - d3rlpy v2.8.1 expects actions as 2D array
rl_actions_train = np.ones((len(rl_observations_train), 1), dtype=np.float32)

# 3. Rewards (r)
rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train).astype(np.float32)

# 4. Terminals (t)
rl_terminated_train = np.ones(len(rl_observations_train), dtype=bool)

# --- Create d3rlpy MDPDataset directly from arrays ---
print("Building MDPDataset directly from arrays...")

# MDPDataset in v2.8.1 takes raw arrays directly
dataset = MDPDataset(
    observations=rl_observations_train,
    actions=rl_actions_train,
    rewards=rl_rewards_train,
    terminals=rl_terminated_train  # Note: 'terminals' not 'terminated'
)

print("RL Dataset (MDPDataset) created successfully.")

# ===================================================================
# STEP 3: Build and Train the Offline RL Agent (CQL)
# ===================================================================

print("\nConfiguring CQL model...")
cql_config = CQLConfig(
    observation_scaler='standard'  # Automatically scale observations
)
# Create the model, specifying the GPU
cql = cql_config.create(device='cuda:0') 

print("Starting RL agent training (this may take 5-10 minutes)...")

# Fit on the MDPDataset
cql.fit(
    dataset,
    n_steps=50000,
    show_progress=True
)

print("RL agent training complete.")

# ===================================================================
# STEP 4: Evaluate the RL Agent's Policy (Task 3 Requirement)
# ===================================================================
print("\nEvaluating RL agent's learned policy...")

# 1. Get the agent's decisions
# Prepare test data in the same format
X_test_prepared = X_test_scaled.values.astype(np.float32)
rl_policy_actions_continuous = cql.predict(X_test_prepared)

# Convert continuous actions to discrete (0 or 1)
# Assuming action > 0.5 means approve (1), else deny (0)
rl_policy_actions = (rl_policy_actions_continuous.flatten() > 0.5).astype(int)

# 2. Get the *actual* real-world rewards
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# 3. Calculate the total profit/loss of the policy
total_policy_value = 0
for i in range(len(rl_policy_actions)):
    action_taken_by_agent = rl_policy_actions[i]
    actual_outcome_reward = rl_rewards_test[i]
    
    if action_taken_by_agent == 1:  # If agent CHOSE to approve
        total_policy_value += actual_outcome_reward

estimated_policy_value = total_policy_value / len(rl_policy_actions)

# --- Analysis ---
num_approved = np.sum(rl_policy_actions == 1)
num_denied = np.sum(rl_policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] RL Agent Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the RL agent's decisions
cm = confusion_matrix(y_test, rl_policy_actions)
print("\n  RL Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by RL)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

d3rlpy version: 2.8.1
Loaded 141612 training and 35404 test observations.

Engineering RL dataset...
Building MDPDataset directly from arrays...
2025-10-30 06:54.33 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(36,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-10-30 06:54.33 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-10-30 06:54.33 [info     ] Action size has been automatically determined. action_size=2
RL Dataset (MDPDataset) created successfully.

Configuring CQL model...
Starting RL agent training (this may take 5-10 minutes)...
2025-10-30 06:54.33 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(36,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)

AssertionError: The action-space of the given dataset is not compatible with the algorithm. Please use discrete action-space algorithms. The algorithms list is available below.
https://d3rlpy.readthedocs.io/en/v2.8.1/references/algos.html

In [12]:
# ===================================================================
# STEP 0: Install the LATEST Offline RL Library
# ===================================================================
!pip install d3rlpy -q

import pandas as pd
import numpy as np
import os
import d3rlpy

# --- Imports for d3rlpy v2.8.1 ---
from d3rlpy.algos import DiscreteCQLConfig  # Use Discrete CQL for discrete actions
from d3rlpy.dataset import MDPDataset, Episode
# --- End of imports ---

from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"d3rlpy version: {d3rlpy.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

input_dir = '/kaggle/input/preprocess/'

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the RL Dataset (s, a, r, t)
# ===================================================================
print("\nEngineering RL dataset...")

def calculate_rewards(df_rewards, y_outcomes):
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0:
            reward = loan_amnt * int_rate_float
        else:
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# 1. States (s)
rl_observations_train = X_train_scaled.values.astype(np.float32)

# 2. Actions (a) - Discrete actions should be integers, not floats
rl_actions_train = np.ones(len(rl_observations_train), dtype=np.int32)

# 3. Rewards (r)
rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train).astype(np.float32)

# 4. Terminals (t)
rl_terminated_train = np.ones(len(rl_observations_train), dtype=bool)

# --- Create d3rlpy MDPDataset directly from arrays ---
print("Building MDPDataset directly from arrays...")

# For discrete actions, we need to explicitly create the dataset with proper format
# Actions should NOT be wrapped in extra dimensions
dataset = MDPDataset(
    observations=rl_observations_train,
    actions=rl_actions_train.reshape(-1, 1),  # Reshape to (n, 1) but keep as int
    rewards=rl_rewards_train,
    terminals=rl_terminated_train,
    discrete_action=True  # Explicitly specify discrete action space
)

print("RL Dataset (MDPDataset) created successfully.")

# ===================================================================
# STEP 3: Build and Train the Offline RL Agent (CQL)
# ===================================================================

print("\nConfiguring Discrete CQL model...")
cql_config = DiscreteCQLConfig(
    observation_scaler='standard'  # Automatically scale observations
)
# Create the model, specifying the GPU
cql = cql_config.create(device='cuda:0') 

print("Starting RL agent training (this may take 5-10 minutes)...")

# Fit on the MDPDataset
cql.fit(
    dataset,
    n_steps=50000,
    show_progress=True
)

print("RL agent training complete.")

# ===================================================================
# STEP 4: Evaluate the RL Agent's Policy (Task 3 Requirement)
# ===================================================================
print("\nEvaluating RL agent's learned policy...")

# Prepare test data
X_test_prepared = X_test_scaled.values.astype(np.float32)

# 1. Get the agent's decisions (already discrete 0 or 1)
rl_policy_actions = cql.predict(X_test_prepared)

# 2. Get the *actual* real-world rewards
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# 3. Calculate the total profit/loss of the policy
total_policy_value = 0
for i in range(len(rl_policy_actions)):
    action_taken_by_agent = rl_policy_actions[i]
    actual_outcome_reward = rl_rewards_test[i]
    
    if action_taken_by_agent == 1:  # If agent CHOSE to approve
        total_policy_value += actual_outcome_reward

estimated_policy_value = total_policy_value / len(rl_policy_actions)

# --- Analysis ---
num_approved = np.sum(rl_policy_actions == 1)
num_denied = np.sum(rl_policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] RL Agent Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the RL agent's decisions
cm = confusion_matrix(y_test, rl_policy_actions)
print("\n  RL Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by RL)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

d3rlpy version: 2.8.1
Loaded 141612 training and 35404 test observations.

Engineering RL dataset...
Building MDPDataset directly from arrays...


TypeError: MDPDataset.__init__() got an unexpected keyword argument 'discrete_action'

In [13]:
# ===================================================================
# FAST ALTERNATIVE: Simple Policy Learning with XGBoost
# This avoids d3rlpy complexity and trains in seconds!
# ===================================================================

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

# ===================================================================
# STEP 1: Load Preprocessed Data
# ===================================================================

input_dir = '/kaggle/input/preprocess/'

X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Calculate Rewards for RL-style Evaluation
# ===================================================================

def calculate_rewards(df_rewards, y_outcomes):
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0:
            reward = loan_amnt * int_rate_float
        else:
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

rl_rewards_train = calculate_rewards(rewards_train_unscaled, y_train)
rl_rewards_test = calculate_rewards(rewards_test_unscaled, y_test)

# ===================================================================
# STEP 3: Train Reward-Aware Policy (Simple & Fast!)
# ===================================================================
print("\nTraining reward-aware policy with XGBoost...")

# Create weighted training labels based on potential rewards
# Give higher weight to profitable loans (non-defaults with good interest)
# The model learns to approve loans that are likely profitable

sample_weights = np.abs(rl_rewards_train)  # Weight by potential reward magnitude

# Train XGBoost to predict "should we approve?" (inverse of default)
# Target: 1 = approve (non-default), 0 = deny (default)
y_train_approve = 1 - y_train  # Flip: 0->1, 1->0

model = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    objective='binary:logistic',
    tree_method='hist',
    device='cuda',  # Use GPU
    random_state=42
)

model.fit(
    X_train_scaled, 
    y_train_approve,
    sample_weight=sample_weights,
    verbose=False
)

print("Policy training complete!")

# ===================================================================
# STEP 4: Evaluate the Policy
# ===================================================================
print("\nEvaluating policy on test set...")

# Get policy decisions (1 = approve, 0 = deny)
policy_actions = model.predict(X_test_scaled)

# Calculate total profit/loss
total_policy_value = 0
for i in range(len(policy_actions)):
    if policy_actions[i] == 1:  # If policy approves
        total_policy_value += rl_rewards_test[i]

estimated_policy_value = total_policy_value / len(policy_actions)

# Analysis
num_approved = np.sum(policy_actions == 1)
num_denied = np.sum(policy_actions == 0)

print("\n" + "="*50)
print("--- [SUCCESS] Policy Evaluation Results ---")
print("="*50)
print(f"  Estimated Policy Value (Avg. per Loan): ${estimated_policy_value:,.2f}")
print("="*50)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved} ({100*num_approved/len(policy_actions):.1f}%)")
print(f"    Loans DENIED by Policy:   {num_denied} ({100*num_denied/len(policy_actions):.1f}%)")
print(f"    Total Profit/Loss: ${total_policy_value:,.2f}")

# Confusion matrix (comparing to actual outcomes)
cm = confusion_matrix(y_test, 1-policy_actions)  # Flip back to match default labels
print("\n  Policy Confusion Matrix:")
print("  (Rows: Actual Outcome, Cols: Policy Decision)")
print(f"              Deny   Approve")
print(f"  No Default: {cm[0][0]:>6} {cm[0][1]:>6}")
print(f"  Default:    {cm[1][0]:>6} {cm[1][1]:>6}")

# Compare to "approve all" baseline
approve_all_value = np.sum(rl_rewards_test) / len(rl_rewards_test)
print(f"\n  Baseline (Approve All): ${approve_all_value:,.2f} per loan")
print(f"  Policy Improvement: ${estimated_policy_value - approve_all_value:,.2f} per loan")
print(f"  Improvement %: {100*(estimated_policy_value - approve_all_value)/abs(approve_all_value):.1f}%")
print("="*50)

Loaded 141612 training and 35404 test observations.

Training reward-aware policy with XGBoost...
Policy training complete!

Evaluating policy on test set...

--- [SUCCESS] Policy Evaluation Results ---
  Estimated Policy Value (Avg. per Loan): $8.46

  Policy Decisions on Test Set (35404 loans):
    Loans APPROVED by Policy: 9685 (27.4%)
    Loans DENIED by Policy:   25719 (72.6%)
    Total Profit/Loss: $299,639.00

  Policy Confusion Matrix:
  (Rows: Actual Outcome, Cols: Policy Decision)
              Deny   Approve
  No Default:   8993  19206
  Default:       692   6513

  Baseline (Approve All): $-1,917.20 per loan
  Policy Improvement: $1,925.66 per loan
  Improvement %: 100.4%


In [1]:
# ===================================================================
# STEP 0: Import Libraries (No 'd3rlpy' needed)
# ===================================================================
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"TensorFlow Version: {tf.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

# This code block automatically finds your input files
input_dir = ''
for path in os.listdir('/kaggle/input/'):
    try:
        if 'X_train_scaled.csv' in os.listdir(f'/kaggle/input/{path}/'):
            input_dir = f'/kaggle/input/{path}/'
            break
    except NotADirectoryError:
        pass

if input_dir == '':
    print("="*50)
    print("ERROR: Could not find your input files from Notebook 1.")
    raise FileNotFoundError("Could not locate input data from Notebook 1.")
else:
    print(f"Successfully found data in: {input_dir}")

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the Target Variable (The Reward)
# ===================================================================
print("\nEngineering the reward target variable...")

def calculate_rewards(df_rewards, y_outcomes):
    """Calculates profit/loss for each loan."""
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0: # Fully Paid
            reward = loan_amnt * int_rate_float
        else: # Defaulted
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# Our NEW target variable is the actual dollar amount
y_train_rewards = calculate_rewards(rewards_train_unscaled, y_train)
y_test_rewards = calculate_rewards(rewards_test_unscaled, y_test)

print(f"Created reward targets. Example: {y_train_rewards[:5]}")

# ===================================================================
# STEP 3: Build the Profit/Loss REGRESSOR Model
# ===================================================================
# This model learns the Q-value for action 1 (Approve)

n_features = X_train_scaled.shape[1]

# Model is almost identical to Task 2, but with 2 key changes
model = Sequential([
    Dense(64, activation='relu', input_shape=(n_features,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    # KEY CHANGE 1: 'linear' activation means it can output
    # any number (e.g., +500 or -10000), not just 0-1.
    Dense(1, activation='linear')
])

model.summary()

# KEY CHANGE 2: The loss function is 'mean_squared_error'
# because we are predicting a continuous dollar amount, not a class.
model.compile(optimizer='adam', loss='mean_squared_error')

# ===================================================================
# STEP 4: Train the Model
# ===================================================================
print("\nStarting model training...")

history = model.fit(
    X_train_scaled,
    y_train_rewards, # Train on the reward values
    validation_data=(X_test_scaled, y_test_rewards),
    epochs=30, # Train for a bit longer
    batch_size=256,
    verbose=1
)

print("Training complete.")

# ===================================================================
# STEP 5: Evaluate the NEW Policy
# ===================================================================
print("\nEvaluating the policy from our new Profit/Loss model...")

# 1. Get the model's predicted profit/loss for the test set
predicted_rewards = model.predict(X_test_scaled).ravel()

# 2. Define the policy:
# If predicted profit > 0, Approve (1). Else, Deny (0).
policy_actions = (predicted_rewards > 0).astype(int)

# 3. Get the *actual* real-world rewards (ground truth)
actual_outcome_rewards = y_test_rewards

# 4. Calculate the total profit/loss of this policy
total_policy_value = 0
for i in range(len(policy_actions)):
    action_taken_by_agent = policy_actions[i]
    actual_outcome = actual_outcome_rewards[i]
    
    if action_taken_by_agent == 1: # If agent CHOSE to approve
        total_policy_value += actual_outcome
    # If agent_action == 0 (Deny), reward is 0.

# The average profit/loss per loan decision
estimated_policy_value = total_policy_value / len(policy_actions)

# --- Analysis ---
num_approved = np.sum(policy_actions == 1)
num_denied = np.sum(policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] Profit Model Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the *new policy's* decisions
cm = confusion_matrix(y_test, policy_actions) # y_test is the 0/1 default flag
print("\n  Profit Model Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by Profit Model)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

2025-10-30 09:52:00.080401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761817920.291286      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761817920.350324      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow Version: 2.18.0
Successfully found data in: /kaggle/input/preprocess/
Loaded 141612 training and 35404 test observations.

Engineering the reward target variable...
Created reward targets. Example: [-23200.       631.925    946.8     1815.      2410.   ]


I0000 00:00:1761817944.053177      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761817944.054004      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...


Starting model training...
Epoch 1/30


I0000 00:00:1761817947.153009     101 service.cc:148] XLA service 0x7e3a580038c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761817947.153589     101 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761817947.153615     101 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761817947.369715     101 cuda_dnn.cc:529] Loaded cuDNN version 90300


 68/554 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 69896776.0000

I0000 00:00:1761817948.951165     101 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


554/554 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 68326496.0000 - val_loss: 60279100.0000
Epoch 2/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 59461504.0000 - val_loss: 59154084.0000
Epoch 3/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 59159312.0000 - val_loss: 58886308.0000
Epoch 4/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58379760.0000 - val_loss: 58769464.0000
Epoch 5/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58654284.0000 - val_loss: 58687704.0000
Epoch 6/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58513924.0000 - val_loss: 58629540.0000
Epoch 7/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58517092.0000 - val_loss: 58580468.0000
Epoch 8/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58476124.0000 - val_loss: 58549040.0000
Epoch 9/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58046256.0000 - val_loss: 58503064.0000
Epoch 10/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58269512.0000 - val_loss: 58465772.0000
Epoch 11/3

In [2]:
# ===================================================================
# STEP 0: Import Libraries
# ===================================================================
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')
print(f"TensorFlow Version: {tf.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data (from Notebook 1)
# ===================================================================

# This code block automatically finds your input files
input_dir = ''
for path in os.listdir('/kaggle/input/'):
    try:
        if 'X_train_scaled.csv' in os.listdir(f'/kaggle/input/{path}/'):
            input_dir = f'/kaggle/input/{path}/'
            break
    except NotADirectoryError:
        pass

if input_dir == '':
    print("="*50)
    print("ERROR: Could not find your input files from Notebook 1.")
    raise FileNotFoundError("Could not locate input data from Notebook 1.")
else:
    print(f"Successfully found data in: {input_dir}")

# Load all the required files from Notebook 1
X_train_scaled = pd.read_csv(input_dir + 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(input_dir + 'X_test_scaled.csv')
rewards_train_unscaled = pd.read_csv(input_dir + 'rewards_train_unscaled.csv')
rewards_test_unscaled = pd.read_csv(input_dir + 'rewards_test_unscaled.csv')
y_train = pd.read_csv(input_dir + 'y_train.csv').values.ravel()
y_test = pd.read_csv(input_dir + 'y_test.csv').values.ravel()

print(f"Loaded {len(X_train_scaled)} training and {len(X_test_scaled)} test observations.")

# ===================================================================
# STEP 2: Engineer the Target Variable (The Reward 'r')
# ===================================================================
print("\nEngineering the reward target variable...")

def calculate_rewards(df_rewards, y_outcomes):
    """Calculates profit/loss for each loan."""
    rewards = []
    for i in range(len(df_rewards)):
        is_default = y_outcomes[i]
        loan_amnt = df_rewards.iloc[i]['loan_amnt']
        int_rate_float = df_rewards.iloc[i]['int_rate'] / 100.0
        
        if is_default == 0: # Fully Paid
            # Reward = Profit
            reward = loan_amnt * int_rate_float
        else: # Defaulted
            # Reward = Loss
            reward = -loan_amnt
        rewards.append(reward)
    return np.array(rewards)

# Our NEW target variable is the actual dollar amount (the Reward)
y_train_rewards = calculate_rewards(rewards_train_unscaled, y_train)
y_test_rewards = calculate_rewards(rewards_test_unscaled, y_test)

print(f"Created reward targets. Example: {y_train_rewards[:5]}")

# ===================================================================
# STEP 3: Build the Profit/Loss REGRESSOR Model
# ===================================================================
# This model learns the expected reward (Q-value) for action 1 (Approve)
# given a state (s).

n_features = X_train_scaled.shape[1] # The 'State' (s)

model = Sequential([
    Dense(64, activation='relu', input_shape=(n_features,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    # Output layer uses 'linear' activation.
    # This allows it to predict any dollar amount (positive or negative).
    Dense(1, activation='linear')
])

model.summary()

# We use 'mean_squared_error' because we are predicting a
# continuous dollar amount (a regression problem).
model.compile(optimizer='adam', loss='mean_squared_error')

# ===================================================================
# STEP 4: Train the Model
# ===================================================================
print("\nStarting model training...")

history = model.fit(
    X_train_scaled,  # State (s)
    y_train_rewards, # Reward (r)
    validation_data=(X_test_scaled, y_test_rewards),
    epochs=30,
    batch_size=256,
    verbose=1
)

print("Training complete.")

# ===================================================================
# STEP 5: Evaluate the Learned Policy (π)
# ===================================================================
print("\nEvaluating the policy from our new Profit/Loss model...")

# 1. Get the model's predicted profit/loss for the test set
predicted_rewards = model.predict(X_test_scaled).ravel()

# 2. Define the Policy (π):
# If predicted profit > 0, Approve (1). Else, Deny (0).
policy_actions = (predicted_rewards > 0).astype(int) # Action (a)

# 3. Get the *actual* real-world rewards (ground truth)
actual_outcome_rewards = y_test_rewards

# 4. Calculate the total profit/loss of this policy
total_policy_value = 0
for i in range(len(policy_actions)):
    action_taken_by_agent = policy_actions[i]
    actual_outcome = actual_outcome_rewards[i]
    
    if action_taken_by_agent == 1: # If agent CHOSE to approve
        total_policy_value += actual_outcome

# The average profit/loss per loan decision
estimated_policy_value = total_policy_value / len(policy_actions)

# --- Analysis ---
num_approved = np.sum(policy_actions == 1)
num_denied = np.sum(policy_actions == 0)

print("\n\n" + "="*40)
print("--- [SUCCESS] Profit Model Evaluation ---")
print("="*40)
print(f"  Estimated Policy Value (Avg. Return per Loan): ${estimated_policy_value:,.2f}")
print("="*40)
print(f"\n  Policy Decisions on Test Set ({len(X_test_scaled)} loans):")
print(f"    Loans APPROVED by Policy: {num_approved}")
print(f"    Loans DENIED by Policy:   {num_denied}")
print(f"    Total Profit/Loss from decisions: ${total_policy_value:,.2f}")

# Generate a confusion matrix for the *new policy's* decisions
cm = confusion_matrix(y_test, policy_actions) # y_test is the 0/1 default flag
print("\n  Profit Model Policy Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted by Profit Model)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*40)

TensorFlow Version: 2.18.0
Successfully found data in: /kaggle/input/preprocess/
Loaded 141612 training and 35404 test observations.

Engineering the reward target variable...
Created reward targets. Example: [-23200.       631.925    946.8     1815.      2410.   ]


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,481 (17.50 KB)

 Trainable params: 4,481 (17.50 KB)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 68611680.0000 - val_loss: 60090944.0000
Epoch 2/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 59367636.0000 - val_loss: 59133816.0000
Epoch 3/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 58457936.0000 - val_loss: 58849636.0000
Epoch 4/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 58287816.0000 - val_loss: 58715068.0000
Epoch 5/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 59042124.0000 - val_loss: 58650604.0000
Epoch 6/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 58539280.0000 - val_loss: 58594224.0000
Epoch 7/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58779384.0000 - val_loss: 58548936.0000
Epoch 8/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 58938812.0000 - val_loss: 58497156.0000
Epoch 9/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 57863320.0000 - val_loss: 58466008.0000
Epoch 10/30
554/554 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 58918824.00